In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import os

In [2]:
# Define paths
train_dir = r"C:\Users\91951\Downloads\trainfood"
test_dir = r"C:\Users\91951\Downloads\testfood"


In [3]:
# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)


In [4]:
# Rescaling for testing
test_datagen = ImageDataGenerator(rescale=1./255)


In [5]:
# Load data
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 3005 images belonging to 4 classes.
Found 815 images belonging to 4 classes.


In [6]:
# Load pre-trained MobileNetV2 base
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze base layers initially


In [7]:
# Build the model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')
])

In [8]:
# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])


In [9]:
# Model summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,468 (9.24 MB)

 Trainable params: 164,484 (642.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [10]:
# Train the model
history = model.fit(train_generator,epochs=15,validation_data=test_generator)


C:\Users\91951\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
94/94 ━━━━━━━━━━━━━━━━━━━━ 132s 1s/step - accuracy: 0.7154 - loss: 0.7706 - val_accuracy: 0.9337 - val_loss: 0.1851
Epoch 2/15
94/94 ━━━━━━━━━━━━━━━━━━━━ 111s 1s/step - accuracy: 0.9276 - loss: 0.2157 - val_accuracy: 0.9497 - val_loss: 0.1646
Epoch 3/15
94/94 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - accuracy: 0.9357 - loss: 0.1802 - val_accuracy: 0.9497 - val_loss: 0.1619
Epoch 4/15
94/94 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - accuracy: 0.9543 - loss: 0.1399 - val_accuracy: 0.9325 - val_loss: 0.2040
Epoch 5/15
94/94 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - accuracy: 0.9517 - loss: 0.1461 - val_accuracy: 0.9558 - val_loss: 0.1464
Epoch 6/15
94/94 ━━━━━━━━━━━━━━━━━━━━ 110s 1s/step - accuracy: 0.9602 - loss: 0.1102 - val_accuracy: 0.9521 - val_loss: 0.1586
Epoch 7/15
94/94 ━━━━━━━━━━━━━━━━━━━━ 104s 1s/step - accuracy: 0.9679 - loss: 0.1007 - val_accuracy: 0.9448 - val_loss: 0.1866
Epoch 8/15
94/94 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - accuracy: 0.9566 - loss: 0.1170 - val_accuracy: 0.9411 - v

In [2]:
# Unfreeze some layers for fine-tuning
base_model.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),  # Lower learning rate
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history_fine = model.fit(
    train_generator,
    epochs=15,  # Few epochs for fine-tuning
    validation_data=test_generator
)

NameError: name 'base_model' is not defined

In [ ]:
# Evaluate the model
test_loss, test_accuracy = model.evaluate(test_generator)
print(f"Test accuracy: {test_accuracy:.4f}")


In [ ]:
# Plot accuracy and loss
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'] + history_fine.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'] + history_fine.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'] + history_fine.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'] + history_fine.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.legend()
plt.show()

In [ ]:
# Save the model
model.save('pretrained_4class_model.h5')
print("Pre-trained model saved as 'pretrained_4class_model.h5'")

In [ ]:
import subprocess
import webbrowser
import os
import time

# Streamlit app code 
app_code = '''
import streamlit as st
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import numpy as np
from PIL import Image

# Load the model
@st.cache_resource
def load_model():
    model = tf.keras.models.load_model('pretrained_4class_model.h5')
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = load_model()

def predict_image(uploaded_file):
    img = Image.open(uploaded_file).convert('RGB')
    img = img.resize((224, 224))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, 0)
    pred = model.predict(img_array)[0]
    class_index = np.argmax(pred)
    class_names = ['Chicken Biryani', 'Chicken Tanduri', 'Crispy Chicken', 'Chicken Curry']  
    class_name = class_names[class_index]
    confidence = pred[class_index]
    return class_name, confidence

# App layout
st.title('Chicken Food Classifier')

# File uploader
uploaded_file = st.file_uploader("Choose an image...", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    # Display image
    image = Image.open(uploaded_file)
    st.image(image, caption='Uploaded Image', use_column_width=True)
    
    # Make prediction
    class_name, confidence = predict_image(uploaded_file)
    
    # Display results
    st.write(f'**Prediction:** {class_name}')
    st.write(f'**Confidence:** {confidence:.2f}')
    progress = min(confidence * 100, 100)
    st.progress(progress / 100)
    st.write(f'{progress:.0f}% Confidence')
'''

# Save the code to app.py
with open('app.py', 'w') as f:
    f.write(app_code)
print("app.py created successfully!")

# Launch Streamlit
process = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.headless', 'true', '--server.port', '8501'])

# Open browser
webbrowser.open('http://localhost:8501')
print("Streamlit app launched! Check your browser at http://localhost:8501")
